# Хакатон: Анализ сайта СберАвтоподписки
## Notebook 4: Реализация и тестирование API

В этом ноутбуке я демонстрирую работу API-сервиса, который принимает данные по визиту пользователя и возвращает предсказание: совершит ли он целевое действие (0 или 1).

API реализован в файле `predict.py` с использованием FastAPI.

### 1. Инструкция по запуску API

Перед тестированием необходимо запустить сервер. Для этого нужно открыть терминал (Anaconda Prompt) и выполнить команду:

```
cd "D:\sber-autosubscription-hackathon"
uvicorn predict:app --reload --port 8000
```

После запуска сервер будет доступен по адресу: http://localhost:8000

Документация API (Swagger UI): http://localhost:8000/docs

### 2. Прямое тестирование модели (без HTTP-запроса)

Для демонстрации я также тестирую модель напрямую — загружаю её и вызываю предсказание из ноутбука.

In [1]:
import pandas as pd
import numpy as np
import joblib
import time

DATA_PATH = r'D:\sber-autosubscription-hackathon'

# Загружаю модель и метаданные
model         = joblib.load(f'{DATA_PATH}\\model.pkl')
features_meta = joblib.load(f'{DATA_PATH}\\features_meta.pkl')

ALL_FEATURES        = features_meta['all_features']
NUM_FEATURES        = features_meta['num_features']
CAT_FEATURES        = features_meta['cat_features']
CAT_FEATURE_INDICES = features_meta['cat_feature_indices']

print('Модель загружена.')
print(f'Количество признаков: {len(ALL_FEATURES)}')
print(f'Числовых: {len(NUM_FEATURES)}, категориальных: {len(CAT_FEATURES)}')

Модель загружена.
Количество признаков: 23
Числовых: 16, категориальных: 7


### 3. Функция подготовки признаков

Реализую ту же логику, что и в predict.py — для демонстрации работы пайплайна предсказания.

In [2]:
def prepare_features(visit: dict) -> pd.DataFrame:
    """
    Принимает словарь с параметрами визита,
    возвращает DataFrame с признаками для модели.
    Все признаки доступны на старте визита — hits_count исключён (data leakage).
    """
    visit_number = int(visit.get('visit_number', 1))
    hour         = int(visit.get('hour', 12))
    weekday      = int(visit.get('weekday', 0))
    month        = int(visit.get('month', 6))
    utm_medium   = str(visit.get('utm_medium', 'unknown'))
    device_os    = str(visit.get('device_os', 'unknown'))
    device_cat   = str(visit.get('device_category', 'mobile'))

    try:
        screen_width = int(str(visit.get('device_screen_resolution', '0')).split('x')[0])
    except Exception:
        screen_width = 0

    row = {
        'visit_number':     visit_number,
        'log_visit_number': np.log1p(visit_number),
        'hour':             hour,
        'weekday':          weekday,
        'month':            month,
        'is_organic':       int(utm_medium in ['organic', 'referral', '(none)']),
        'is_mobile':        int(device_cat == 'mobile'),
        'is_desktop':       int(device_cat == 'desktop'),
        'is_tablet':        int(device_cat == 'tablet'),
        'is_night':         int(hour in range(0, 7)),
        'is_work_hours':    int(hour in range(10, 20)),
        'is_weekend':       int(weekday >= 5),
        'is_first_visit':   int(visit_number == 1),
        'is_returning':     int(visit_number > 3),
        'is_ios':           int(device_os in ['iOS', 'Macintosh']),
        'screen_width':     screen_width,
        'utm_medium':       utm_medium,
        'utm_source':       str(visit.get('utm_source', 'unknown')),
        'device_category':  device_cat,
        'device_os':        device_os,
        'device_browser':   str(visit.get('device_browser', 'unknown')),
        'geo_city':         str(visit.get('geo_city', 'unknown')),
        'geo_country':      str(visit.get('geo_country', 'Russia')),
    }

    df = pd.DataFrame([row])[ALL_FEATURES]
    for col in CAT_FEATURES:
        df[col] = df[col].astype(str)
    for col in NUM_FEATURES:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    return df

print('Функция подготовки признаков определена.')
print('Входные параметры: visit_number, hour, weekday, month, utm_medium, utm_source,')
print('                   device_category, device_os, device_browser,')
print('                   device_screen_resolution, geo_country, geo_city.')
print('hits_count не требуется — модель использует только признаки, доступные на старте визита.')

Функция подготовки признаков определена.
Входные параметры: visit_number, hour, weekday, month, utm_medium, utm_source,
                   device_category, device_os, device_browser,
                   device_screen_resolution, geo_country, geo_city.
hits_count не требуется — модель использует только признаки, доступные на старте визита.


### 4. Тестирование на трёх примерах

Проверяю предсказания на трёх типичных профилях пользователей.

In [3]:
test_cases = [
    {
        'name': 'Активный пользователь (высокая вероятность конверсии)',
        'data': {
            'visit_number': 5,
            'hour': 14,
            'weekday': 2,
            'month': 3,
            'utm_medium': 'organic',
            'utm_source': 'google',
            'device_category': 'desktop',
            'device_os': 'Windows',
            'device_browser': 'Chrome',
            'device_screen_resolution': '1920x1080',
            'geo_country': 'Russia',
            'geo_city': 'Moscow'
        }
    },
    {
        'name': 'Новый мобильный пользователь (низкая вероятность)',
        'data': {
            'visit_number': 1,
            'hour': 20,
            'weekday': 6,
            'month': 11,
            'utm_medium': 'banner',
            'utm_source': 'unknown',
            'device_category': 'mobile',
            'device_os': 'Android',
            'device_browser': 'Chrome',
            'device_screen_resolution': '360x720',
            'geo_country': 'Russia',
            'geo_city': 'Novosibirsk'
        }
    },
    {
        'name': 'Ночной iOS пользователь',
        'data': {
            'visit_number': 3,
            'hour': 2,
            'weekday': 5,
            'month': 5,
            'utm_medium': 'organic',
            'utm_source': 'google',
            'device_category': 'mobile',
            'device_os': 'iOS',
            'device_browser': 'Safari',
            'device_screen_resolution': '390x844',
            'geo_country': 'Russia',
            'geo_city': 'Saint Petersburg'
        }
    }
]

print('=' * 60)
print('РЕЗУЛЬТАТЫ ПРЕДСКАЗАНИЙ')
print('=' * 60)

for case in test_cases:
    start = time.time()
    df    = prepare_features(case['data'])
    pred  = int(model.predict(df)[0])
    proba = float(model.predict_proba(df)[0][1])
    elapsed = round(time.time() - start, 4)

    print(f'\nПрофиль: {case["name"]}')
    print(f'  Предсказание    : {pred} ({"целевое действие" if pred == 1 else "нет целевого действия"})')
    print(f'  Вероятность     : {proba:.4f}')
    print(f'  Время ответа    : {elapsed} сек')

РЕЗУЛЬТАТЫ ПРЕДСКАЗАНИЙ

Профиль: Активный пользователь (высокая вероятность конверсии)
  Предсказание    : 1 (целевое действие)
  Вероятность     : 0.7433
  Время ответа    : 0.0096 сек

Профиль: Новый мобильный пользователь (низкая вероятность)
  Предсказание    : 0 (нет целевого действия)
  Вероятность     : 0.1920
  Время ответа    : 0.0069 сек

Профиль: Ночной iOS пользователь
  Предсказание    : 0 (нет целевого действия)
  Вероятность     : 0.4920
  Время ответа    : 0.0063 сек


### 5. Проверка времени ответа

Согласно критериям оценки, время ответа API должно быть не более 3 секунд. Проверяю на 1000 запросах.

In [4]:
import time

sample_visit = test_cases[0]['data']
n_requests   = 1000
times        = []

for _ in range(n_requests):
    start = time.time()
    df    = prepare_features(sample_visit)
    _     = model.predict(df)[0]
    times.append(time.time() - start)

times = np.array(times)

print(f'Тест на {n_requests} запросах:')
print(f'  Среднее время ответа  : {times.mean()*1000:.2f} мс')
print(f'  Медиана               : {np.median(times)*1000:.2f} мс')
print(f'  Максимум              : {times.max()*1000:.2f} мс')
print(f'  99-й перцентиль       : {np.percentile(times, 99)*1000:.2f} мс')
print(f'\nКритерий <= 3000 мс: {"ВЫПОЛНЕН" if times.max() < 3.0 else "НЕ ВЫПОЛНЕН"}')

Тест на 1000 запросах:
  Среднее время ответа  : 5.27 мс
  Медиана               : 5.14 мс
  Максимум              : 8.77 мс
  99-й перцентиль       : 6.46 мс

Критерий <= 3000 мс: ВЫПОЛНЕН


### 6. Структура API

Описываю реализованные эндпоинты API.

In [5]:
print('=' * 60)
print('СТРУКТУРА API (predict.py)')
print('=' * 60)
print()
print('Базовый URL: http://localhost:8000')
print()
print('Эндпоинты:')
print('  GET  /              — описание сервиса')
print('  GET  /health        — проверка работоспособности')
print('  POST /predict       — предсказание: возвращает 0 или 1')
print('  POST /predict_proba — вероятность целевого действия')
print()
print('Входные параметры (JSON):')
print('  visit_number, hour, weekday, month')
print('  utm_medium, utm_source')
print('  device_category, device_os, device_browser, device_screen_resolution')
print('  geo_country, geo_city')
print()
print('Пример запроса:')
print('''  curl -X POST http://localhost:8000/predict \\''')
print('''       -H "Content-Type: application/json" \\''')
print('''       -d \'{"visit_number": 5, "utm_medium": "organic",''')
print('''            "device_category": "desktop", "geo_city": "Moscow"}\' ''')
print()
print('Пример ответа:')
print('  {"prediction": 1, "label": "целевое действие", "response_time": "0.003 сек"}')
print()
print('Notebook 4 (API) завершён.')

СТРУКТУРА API (predict.py)

Базовый URL: http://localhost:8000

Эндпоинты:
  GET  /              — описание сервиса
  GET  /health        — проверка работоспособности
  POST /predict       — предсказание: возвращает 0 или 1
  POST /predict_proba — вероятность целевого действия

Входные параметры (JSON):
  visit_number, hour, weekday, month
  utm_medium, utm_source
  device_category, device_os, device_browser, device_screen_resolution
  geo_country, geo_city

Пример запроса:
  curl -X POST http://localhost:8000/predict \
       -H "Content-Type: application/json" \
       -d '{"visit_number": 5, "utm_medium": "organic",
            "device_category": "desktop", "geo_city": "Moscow"}' 

Пример ответа:
  {"prediction": 1, "label": "целевое действие", "response_time": "0.003 сек"}

Notebook 4 (API) завершён.
